In [3]:
import sys
sys.path.append('/home/gsf/Reid/logs/logok/python-okx/')

import pandas as pd
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
from tqdm import tqdm


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
from __future__ import annotations
import os
import sys
from collections import defaultdict
from typing import Dict, List, DefaultDict
import math

class Order:
    __slots__ = ("order_id", "side", "size", "price")

    def __init__(self, order_id: int, side: str, size: int, price: float):
        self.order_id = order_id
        self.side = side          # 'b' or 's'
        self.size = size
        self.price = price


class OrderBook:
    def __init__(self) -> None:
        # 订单映射
        self.order_map: Dict[int, Order] = {}

        # 买盘：价格从高到低   卖盘：价格从低到高
        # 用 defaultdict(list) 存储同一价格下的 order_id 列表
        self.bids: DefaultDict[float, List[int]] = defaultdict(list)
        self.asks: DefaultDict[float, List[int]] = defaultdict(list)

    # -------------------- 基本操作 --------------------
    def new_order(self, order_id: int, side: str, size: int, price: float) -> None:
        """新增订单"""
        order = Order(order_id, side, size, price)
        self.order_map[order_id] = order
        if side == "b":
            self.bids[price].append(order_id)
        else:
            self.asks[price].append(order_id)

    def reduce_order(self, order_id: int, new_size: int) -> None:
        """减少订单数量"""
        if order_id not in self.order_map:
            return
        self.order_map[order_id].size = new_size
        if new_size == 0:
            self.delete_order(order_id)

    def modify_order(self, old_order_id: int, order_id: int,
                     side: str, size: int, price: float) -> None:
        """修改订单：先删后加"""
        self.delete_order(old_order_id)
        self.new_order(order_id, side, size, price)

    def delete_order(self, order_id: int) -> None:
        """删除订单"""
        if order_id not in self.order_map:
            return
        order = self.order_map.pop(order_id)
        price, side = order.price, order.side
        if side == "b":
            vec = self.bids[price]
            vec.remove(order_id)
            if not vec:
                del self.bids[price]
        else:
            vec = self.asks[price]
            vec.remove(order_id)
            if not vec:
                del self.asks[price]

    # -------------------- 查询接口 --------------------
    def get_num_levels(self, side: str) -> int:
        """返回价格档位数"""
        return len(self.bids) if side == "b" else len(self.asks)

    def _sorted_prices(self, side: str) -> List[float]:
        """返回已排序的价格列表（买盘降序，卖盘升序）"""
        if side == "b":
            return sorted(self.bids.keys(), reverse=True)
        return sorted(self.asks.keys())

    def get_level_price(self, side: str, level: int) -> float:
        """返回第 level 档价格（level 0 是 top-of-book）"""
        if level < 0:
            return math.nan
        prices = self._sorted_prices(side)
        if level >= len(prices):
            return math.nan
        return prices[level]

    def get_level_size(self, side: str, level: int) -> int:
        """返回第 level 档总股数"""
        if level < 0:
            return 0
        prices = self._sorted_prices(side)
        if level >= len(prices):
            return 0
        price = prices[level]
        vec = self.bids[price] if side == "b" else self.asks[price]
        return sum(self.order_map[oid].size for oid in vec)

    def get_level_order_count(self, side: str, level: int) -> int:
        """返回第 level 档订单笔数"""
        if level < 0:
            return 0
        prices = self._sorted_prices(side)
        if level >= len(prices):
            return 0
        price = prices[level]
        return len(self.bids[price] if side == "b" else self.asks[price])

    def get_level_size_before(self, side: str, price: float, order_id: int) -> int:
        # 选择正确的盘口 (bids 或 asks)
        book_side = self.bids if side == "b" else self.asks

        # 检查价格档位是否存在
        if price not in book_side:
            return 0
        
        order_list = book_side[price]
        try:
            # 找到目标订单在列表中的索引位置
            index = order_list.index(order_id)
        except ValueError:
            # 如果订单ID不在这个价格档位的列表中，说明有问题，返回0
            return 0
        
        # 累加索引位置之前所有订单的 size
        total_size = 0
        for oid in order_list[:index]:
            total_size += self.order_map[oid].size
        
        return total_size

file_df = pd.read_csv('0700hk.csv', index_col=0)
for key, row in file_df[((file_df['MSG']=='DEL') | (file_df['MSG']=='REPLACE')) & (file_df['NEXTMSG']!='TRD')]:
    
    book, order_id, side = book_dict[key], row['ORDER_ID'], row['SIDE']
    price = book.order_map[order_id].price
    if side=='B':
         ##统计book.bid[price]队列在order_id之前入队的订单(增加一个book类函数叫get_level_size_before) 记为cxl_qb, 统计这个book.bid[price] 队列全部的订单 记为cxl_qall
        cxl_prior_B = 1 - cxl_qb/cxl_qall

In [ ]:
from copy import deepcopy as copy

In [ ]:
book_dict = {}
book = OrderBook()
# 逐行读指令
for key,row in file_df.iterrows():
    instr = row['MSG']
    if instr == "ADDMOD":          # new
        side, oid, size, price = row['SIDE'], row['ORDER_ID'], row['SIZE'], row['PRICE']
        book.new_order(int(oid), side, int(size), float(price))
    elif instr == "REPLACE":        # reduce
        _, oid, size = row['ORDER_ID'], row['SIZE']
        book.reduce_order(int(oid), int(size))
    # elif instr == "REPLACE":        # modify
    #     _, old_oid, oid, side, size, price = parts
    #     book.modify_order(int(old_oid), int(oid), side, int(size), float(price))
    elif instr == "DEL":        # delete
        _, oid = row['OID']
        book.delete_order(int(oid))
    book_dict['key'] = copy(book)


In [ ]:
for key, row in file_df[((file_df['MSG']=='DEL') | (file_df['MSG']=='REPLACE')) & (file_df['NEXTMSG']!='TRD')].iterrows(): # 注意这里用 .iterrows()
    book = book_dict[key] 
    order_id = row['ORDER_ID']
    side = row['SIDE']
    if order_id not in book.order_map:
        continue 
    price = book.order_map[order_id].price
    cxl_qb = book.get_level_size_before(side, price, order_id)
    cxl_qall = book.get_level_total_size(side, price)
    if cxl_qall > 0:
        cxl_prior = 1.0 - (cxl_qb / cxl_qall)
    else:
        cxl_prior = 0.0 # 或者 NaN，如果这是一个无效情况
    row['cxl_prior'] = cxl_prior

In [ ]:
## operators.
    if instr == "p":        # getLevelPrice
        _, side, level = parts
        price = book.get_level_price(side, int(level))
        print(f"{price:.2f}" if not math.isnan(price) else "nan", file=writer)
    elif instr == "s":        # getLevelSize
        _, side, level = parts
        print(book.get_level_size(side, int(level)), file=writer)
    elif instr == "l":        # getNumLevels
        _, side = parts
        print(book.get_num_levels(side), file=writer)
    elif instr == "c":        # getLevelOrderCount
        _, side, level = parts
        print(book.get_level_order_count(side, int(level)), file=writer)
    else:
        print("invalid input", file=writer)
